# Step 0: Import requisite libraries

In [692]:
import math
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn import preprocessing, ensemble, tree
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import matplotlib as plt
import seaborn

remove_excess = False
threshold = 0.5

# Step 1: Import and cleaning of data

In [693]:
df_raw = pd.read_csv("Baza_customer_Telecom_v2.csv")

# Reducing number of negative samples
if remove_excess:
    df_pos = df_raw[df_raw["CHURN"] == "Yes"]
    df_neg = df_raw[df_raw["CHURN"] == "No"]

    neg_pos_ratio = 7/3
    neg_needed = int(len(df_pos) * neg_pos_ratio)

    df_nsampled = df_neg.sample(n=neg_needed, random_state=42)
    df = pd.concat([df_pos, df_nsampled]).sample(frac=1, random_state=42).reset_index(drop=True)
else:
    df = df_raw
df

,PID,CRM_PID_Value_Segment,EffectiveSegment,Billing_ZIP,KA_name,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,CHURN
0,123759242,Bronze,SOHO,6000.0,VM,2,NaN,NaN,2,40.17,0.0,40.17,NaN,No
1,126145737,Bronze,SOHO,6400.0,VM,3,NaN,NaN,3,40.17,0.0,40.17,13.39,No
2,123506355,Bronze,SOHO,6000.0,DI,2,3.0,NaN,5,40.17,0.0,40.17,20.09,No
3,112595585,Bronze,SOHO,4400.0,MT,1,2.0,NaN,3,40.17,0.0,40.17,40.17,No
4,115097935,Iron,SOHO,4000.0,AD,2,1.0,NaN,3,40.17,0.0,40.17,20.09,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8448,115901902,Iron,SOHO,4000.0,VU,2,1.0,NaN,3,56.50,0.0,56.50,28.25,No
8449,115800273,Bronze,SOHO,4006.0,VU,5,NaN,NaN,5,56.50,0.0,56.50,11.30,No
8450,160049889,Bronze,SOHO,4003.0,VU,3,1.0,NaN,4,56.50,0.0,56.50,18.83,No
8451,160095203,Bronze,SOHO,4023.0,VU,3,NaN,NaN,3,56.50,0.0,56.50,18.83,Yes


In [694]:
print(f"Length of dataset before cleaning: {len(df)}")
# Cleaning for PID
df = df.loc[df["PID"].notnull() & df["PID"].str.fullmatch(r"[0-9]*(SMB)?")] # Some PIDs were scientific notation (invalid); valid format is integer optionally followed by "SMB"

# Cleaning for value segment
df = df.loc[df["CRM_PID_Value_Segment"].notnull()]
df["CRM_PID_Value_Segment"] = df["CRM_PID_Value_Segment"].replace("Sliver", "Silver") # Fix assumed typo in one entry

# Cleaning for effective segment
df = df.loc[df["EffectiveSegment"].notnull()]

# Cleaning for ZIP
df = df.loc[df["Billing_ZIP"] % 1 == 0] # Remove non-integer ZIP codes (invalid; includes nulls)

# Cleaning for KA_name
df = df.loc[df["KA_name"].notnull()]
df["KA_name"] = df["KA_name"].replace("AD?", "AD") # Grouped AD? with AD under assumption that unsure attribution is correct

# Cleaning for active_subscribers
df = df.loc[df["Active_subscribers"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for not_active_subscribers
df["Not_Active_subscribers"] = df["Not_Active_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none inactive; minimum non-NaN value is 1)

# Cleaning for suspended_subscribers
df["Suspended_subscribers"] = df["Suspended_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none suspended; minimum non-NaN value is 1)

# Cleaning for total_SUBs
df = df.loc[df["Total_SUBs"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for AvgMobileRevenue
df = df.loc[df["AvgMobileRevenue "].notnull()] # Redundant (no nulls) but included for consistency
df = df.rename(columns={"AvgMobileRevenue ": "AvgMobileRevenue"}) # Remove trailing space from column name

# Cleaning for AvgFIXRevenue
df = df.loc[df["AvgFIXRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for TotalRevenue
df = df.loc[df["TotalRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for ARPU
df = df.loc[df["ARPU"].notnull()]

# Cleaning for CHURN
df = df.loc[df["CHURN"].isin(["No", "Yes"])] # Remove invalid CHURN values (only valid values are "No" and "Yes"); redundant (no invalids) but included for consistency

print(f"Length of dataset after cleaning: {len(df)}") # Small length change, hence valid to discard nulls/invalid entries

# Source - https://stackoverflow.com/a/48035423
# Posted by BENY, modified by community. See post 'Timeline' for change history
# Retrieved 2026-09-02, License - CC BY-SA 4.0

df.apply(lambda x : pd.factorize(x)[0]).corr(method='pearson', min_periods=1)
segment_order = (df.groupby("CRM_PID_Value_Segment")["TotalRevenue"]
                 .mean()
                 .sort_values()
                 .index
                 .tolist())

Length of dataset before cleaning: 8453
Length of dataset after cleaning: 8419


# Step 2: Data Preprocessing

In [695]:
X = df.drop(columns=["CHURN"])
y = df["CHURN"].map({"No": 0, "Yes": 1}) # Convert CHURN to binary values (0 for No, 1 for Yes)

X["PID_is_SMB"] = X["PID"].str.endswith("SMB").astype("string") # Create binary feature indicating if PID ends with "SMB"
X["PID"]=X["PID"].str.replace("SMB", "").astype(int) # Remove "SMB" from PID and convert to integer

# Definition of categorical and numerical features for preprocessing
cat_features = ["CRM_PID_Value_Segment", "EffectiveSegment", "KA_name", "PID_is_SMB"]
num_features = ["Billing_ZIP", "Active_subscribers", "Not_Active_subscribers", "Suspended_subscribers", "Total_SUBs", "AvgMobileRevenue", "AvgFIXRevenue", "TotalRevenue", "ARPU"]

# Selection of features for model training

train_features = cat_features + num_features

class_weights = {0: len(y == 1), 1: len(y == 0)}

# train_features = [
#     "PID_is_SMB",
#     "CRM_PID_Value_Segment", 
#     "Active_subscribers",
#     "AvgMobileRevenue",
#     "TotalRevenue"
#     ]

X = X[train_features]

# Definition of ColumnTransformer for preprocessing
# c_transformer = ColumnTransformer(
#     transformers=[
#         ("categorical", preprocessing.OneHotEncoder(handle_unknown="ignore"), list(set(cat_features) & set(train_features)) ),
#         ("numerical", "passthrough", list(set(num_features) & set(train_features)) )
#     ]
# )


# print(f"Column transformer has categorical features {list(set(cat_features) & set(train_features))} and numerical features {list(set(num_features) & set(train_features))}")


# XGBoost prep
for col in cat_features:
    X[col] = X[col].astype('category')

# Train/test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)
# Model pipeline
clf_p = Pipeline([
    # ("preprocessor", c_transformer),
    # ("classifier", ensemble.RandomForestClassifier(
    #     n_estimators=500,
    #     class_weight="balanced",
    #     n_jobs=-1,
    #     random_state=42))

    # ("classifier", tree.DecisionTreeClassifier(
    #     class_weight="balanced",
    #     splitter="best",
    #     max_depth=None,
    #     random_state=42))

    ("classifier", xgb.XGBClassifier(
        enable_categorical = True,
        tree_method="hist",
        n_estimators = 5000,
        learning_rate = 0.03,
        max_depth = 3,
        min_child_weight = 5,
        subsample = 0.8,
        colsample_bytree = 0.8,
        reg_alpha = 0.0,
        reg_lambda = 5,
        objective = 'binary:logistic',
        eval_metric = 'aucpr',
        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum(),
        early_stopping_rounds = 100,
        random_state = 42
    ))
])


# Fit model
clf_p.fit(X_train, y_train, classifier__eval_set = [(X_valid, y_valid)], classifier__verbose = False)

# Predict on test set
y_pred_p = clf_p.predict_proba(X_test)
t_df = pd.DataFrame(X_test)
t_df["Predicted_Churn_Probability"] = y_pred_p[:, 1] # Get probabilities for positive class (CHURN = Yes)
t_df["Predicted_Churn"] = (t_df["Predicted_Churn_Probability"] >= threshold).astype(int) # Convert probabilities to binary predictions using threshold of 0.5
t_df["Actual_Churn"] = y_test.values # Add actual CHURN values for comparison
t_df["Prediction_type"] = np.select(
    [
        (t_df["Predicted_Churn"] == 1) & (t_df["Actual_Churn"] == 1),
        (t_df["Predicted_Churn"] == 0) & (t_df["Actual_Churn"] == 0),
        (t_df["Predicted_Churn"] == 1) & (t_df["Actual_Churn"] == 0),
        (t_df["Predicted_Churn"] == 0) & (t_df["Actual_Churn"] == 1)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

# Evaluate model performance
y_pred = y_pred_p[:, 1] >= threshold
print(classification_report(y_test, y_pred))
print("Test confusion matrix with threshold", (str(threshold) + ":\n"), confusion_matrix(y_test, y_pred))
print("Training confusion matrix (to check for overfit) with threshold", (str(threshold) + ":\n"),confusion_matrix(y_train, clf_p.predict(X_train)))
t_df.to_csv("Predicted_Churn_Results.csv", index=False) # Save predictions to CSV for further analysis
t_df[t_df["Prediction_type"] == "True Positive"].sort_values(by="Predicted_Churn_Probability", ascending=False).head(20) # Sort by predicted probability for easier analysis


              precision    recall  f1-score   support

           0       0.95      0.62      0.75      1575
           1       0.08      0.49      0.14       109

    accuracy                           0.61      1684
   macro avg       0.51      0.55      0.44      1684
weighted avg       0.89      0.61      0.71      1684

Test confusion matrix with threshold 0.5:
 [[969 606]
 [ 56  53]]
Training confusion matrix (to check for overfit) with threshold 0.5:
 [[3151 1890]
 [ 124  223]]


,CRM_PID_Value_Segment,EffectiveSegment,KA_name,PID_is_SMB,Billing_ZIP,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,Predicted_Churn_Probability,Predicted_Churn,Actual_Churn,Prediction_type
4390,Gold,VSE,Tatiana Trifonova,False,4002.0,18,2.0,1.0,21,276.67,0.0,276.67,15.37,0.626905,1,1,True Positive
3705,Gold,VSE,Tatiana Trifonova,False,4000.0,13,5.0,0.0,18,202.83,0.0,202.83,15.60,0.623415,1,1,True Positive
3712,Silver,VSE,Tatiana Trifonova,False,4004.0,13,1.0,1.0,15,203.33,0.0,203.33,15.64,0.621739,1,1,True Positive
4664,Gold,VSE,Daniela Stefanova,False,4002.0,15,1.0,0.0,16,316.33,0.0,316.33,21.09,0.620970,1,1,True Positive
4192,Gold,VSE,Daniela Stefanova,False,4210.0,14,1.0,0.0,15,248.50,0.0,248.50,17.75,0.620970,1,1,True Positive
3328,Bronze,SOHO,VU,False,4600.0,6,0.0,0.0,6,174.83,0.0,174.83,29.14,0.620810,1,1,True Positive
4543,Gold,VSE,Daniela Stefanova,False,4023.0,29,0.0,0.0,29,297.33,0.0,297.33,10.25,0.613066,1,1,True Positive
4919,Platinum,VSE,Ginka Vachkova,False,4210.0,20,2.0,1.0,23,381.00,0.0,381.00,19.05,0.606145,1,1,True Positive
4997,Platinum,VSE,Daniela Stefanova,False,4199.0,27,0.0,0.0,27,403.33,0.0,403.33,14.94,0.604845,1,1,True Positive
4388,Gold,VSE,Daniela Stefanova,False,7800.0,31,12.0,0.0,43,276.50,0.0,276.50,8.92,0.592166,1,1,True Positive


In [696]:
t_df[t_df["Prediction_type"] == "False Positive"].sort_values(by="Predicted_Churn_Probability", ascending=False).head(20) # Sort by predicted probability for easier analysis

,CRM_PID_Value_Segment,EffectiveSegment,KA_name,PID_is_SMB,Billing_ZIP,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,Predicted_Churn_Probability,Predicted_Churn,Actual_Churn,Prediction_type
3260,Silver,SOHO,VU,False,4000.0,14,3.0,0.0,17,170.67,0.0,170.67,12.19,0.625675,1,0,False Positive
4518,Gold,VSE,VU,False,4003.0,18,2.0,1.0,21,294.00,0.0,294.00,16.33,0.625638,1,0,False Positive
4544,Gold,VSE,VU,False,4180.0,15,7.0,0.0,22,298.00,0.0,298.00,19.87,0.625638,1,0,False Positive
7300,Gold,VSE,Ginka Vachkova,False,4000.0,28,0.0,1.0,29,280.33,0.0,280.33,10.01,0.624787,1,0,False Positive
4630,Gold,VSE,Tatiana Trifonova,False,4003.0,17,4.0,0.0,21,311.00,0.0,311.00,18.29,0.623686,1,0,False Positive
3306,Silver,SOHO,RJ,False,6800.0,5,0.0,0.0,5,173.17,0.0,173.17,34.63,0.623608,1,0,False Positive
4425,Gold,VSE,Daniela Stefanova,False,4000.0,15,4.0,0.0,19,281.17,0.0,281.17,18.74,0.622647,1,0,False Positive
4295,Gold,VSE,Daniela Stefanova,False,4002.0,16,2.0,0.0,18,262.00,0.0,262.00,16.38,0.622647,1,0,False Positive
7547,Gold,VSE,Ginka Vachkova,False,4021.0,22,6.0,1.0,29,490.17,0.0,490.17,22.28,0.622452,1,0,False Positive
3772,Silver,VSE,VU,False,4271.0,13,5.0,0.0,18,208.33,0.0,208.33,16.03,0.622143,1,0,False Positive


In [698]:
from sklearn.metrics import precision_score, recall_score

# Calculate evaluation metrics separately for each CRM value segment.
# The loop follows segment_order so auc_df keeps the business-defined ordering.
auc_rows = []
for segment in segment_order:
    # Select only test-set rows that belong to the current CRM segment.
    segment_mask = X_test["CRM_PID_Value_Segment"] == segment

    # Select the actual churn labels for this segment using the same mask.
    segment_actual = y_test[segment_mask]

    # y_pred_p contains two probability columns: column 0 for No churn and
    # column 1 for Yes churn. ROC-AUC must use the positive-class probability.
    segment_probability = y_pred_p[segment_mask.to_numpy(), 1]

    # Convert probabilities to binary predictions using the global threshold.
    # Precision and recall depend on this threshold; AUC does not.
    segment_prediction = segment_probability >= threshold

    # ROC-AUC is undefined when a segment contains only one actual class.
    # Store NaN for those segments instead of raising an exception.
    segment_auc = (
        roc_auc_score(segment_actual, segment_probability)
        if segment_actual.nunique() == 2
        else np.nan
    )

    # Store one result row for the current segment. Sample counts provide
    # context because metrics from very small segments are unstable.
    auc_rows.append({
        "Segment": segment,
        "AUC": segment_auc,
        "Precision": precision_score(
            segment_actual,
            segment_prediction,
            zero_division=0,
        ),
        "Recall": recall_score(
            segment_actual,
            segment_prediction,
            zero_division=0,
        ),
        "Samples": int(segment_mask.sum()),
        "Positive_samples": int(segment_actual.sum()),
    })

# Convert the collected rows into a DataFrame for comparison and display.
auc_df = pd.DataFrame(auc_rows)
auc_df

,Segment,AUC,Precision,Recall,Samples,Positive_samples
0,Iron,0.210000,0.000000,0.000000,52,2
1,Bronze,0.524051,0.104348,0.244898,752,49
2,Silver,0.645081,0.071749,0.695652,400,23
3,SE,NaN,0.000000,0.000000,13,0
4,Lead,NaN,0.000000,0.000000,6,0
5,Gold,0.596614,0.072816,0.789474,305,19
6,SME,0.400560,0.108108,0.571429,58,7
7,Platinum,0.490637,0.082192,0.666667,98,9
